In [34]:
import requests
import json
from datetime import datetime, timedelta
from tavily import TavilyClient


In [55]:


def get_earnings_info_simple(ticker):
    """
    Simple function to get complete earnings info with Tavily fallback - just input ticker
    Returns structured output: earning_result.transcript, earning_result.earningdate, future_development
    
    Args:
        ticker (str): Stock ticker symbol
    
    Returns:
        dict: Structured earnings result
    """
    return get_earnings_info_with_fallback(ticker)

def get_earnings_info_with_fallback(ticker, api_key="9dfbbfa29d93f4793f246e8fb5ca5e74", tavily_api_key="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"):
    """
    Get complete earnings information with Tavily fallback + future business development layer
    
    Args:
        ticker (str): Stock ticker symbol
        api_key (str): Financial Modeling Prep API key
        tavily_api_key (str): Tavily API key
    
    Returns:
        dict: Structured earnings result with transcript, earningdate, and future_development
    """
    try:
        print(f"🔍 Getting complete earnings info for {ticker}...")
        
        # Step 1: Try to get latest earnings transcript
        print(f"📄 Step 1: Getting latest earnings transcript...")
        transcript_info = get_latest_earnings_transcript(ticker, api_key)
        
        # Step 2: If transcript fails, use Tavily search as fallback
        if 'error' in transcript_info:
            print(f"⚠️ Earnings transcript not available, using Tavily search fallback...")
            transcript_info = get_earnings_info_tavily_fallback(ticker, tavily_api_key)
        
        # Step 3: Get next earnings date from calendar
        print(f"📅 Step 3: Getting next earnings date...")
        next_earnings_info = get_next_earnings_date(ticker, api_key)
        
        # Step 4: If next earnings fails, try Tavily search for upcoming earnings
        if 'error' in next_earnings_info:
            print(f"⚠️ Next earnings date not available, using Tavily search fallback...")
            next_earnings_info = get_upcoming_earnings_tavily_fallback(ticker, tavily_api_key)
        
        # Step 5: ALWAYS get future business development and strategy planning
        print(f"🚀 Step 5: Getting future business development and strategy planning...")
        future_development = get_future_business_development(ticker, tavily_api_key)
        
        # Create structured result
        earning_result = {
            'ticker': ticker.upper(),
            'transcript': transcript_info.get('content', '') if 'error' not in transcript_info else '',
            'earningdate': next_earnings_info.get('date', None) if 'error' not in next_earnings_info else None,
            'future_development': future_development.get('content', '') if 'error' not in future_development else '',
            'retrieved_at': datetime.now().isoformat(),
            'data_source': 'FMP_API' if 'error' not in transcript_info else 'Tavily_Search'
        }
        
        print(f"✅ Complete earnings info retrieved for {ticker}")
        return earning_result
        
    except Exception as e:
        print(f"❌ Error getting complete earnings info: {e}")
        return {'error': f'Error: {str(e)}'}

def get_future_business_development(ticker, tavily_api_key):
    """
    Get future business development and strategy planning using Tavily search
    Follow exact user prompts - NO current year/quarter
    
    Args:
        ticker (str): Stock ticker symbol
        tavily_api_key (str): Tavily API key
    
    Returns:
        dict: Future business development content
    """
    try:
        print(f"🔍 Getting future business development for {ticker}...")
        
        # Initialize Tavily client
        client = TavilyClient(tavily_api_key)
        
        # Search 1: EXACTLY as user specified
        query1 = f"{ticker} future business development growth plans expansion strategy in this coming quarter and coming year"
        print(f"🚀 Search 1: {query1}")
        
        response1 = client.search(
            query=query1,
            include_answer="advanced",
            topic="finance",
            search_depth="advanced",
            max_results=5
        )
        
        # Search 2: EXACTLY as user specified
        query2 = f"stock: {ticker} current business challenges problems solutions strategy / or business success strategies maintenance plan"
        print(f"🔧 Search 2: {query2}")
        
        response2 = client.search(
            query=query2,
            include_answer="advanced",
            topic="finance",
            search_depth="advanced",
            max_results=5
        )
        
        # Extract content from both searches
        future_development = []
        business_challenges = []
        
        # Process Search 1 results
        if 'results' in response1:
            for result in response1['results']:
                if 'content' in result and result['content']:
                    content = result['content']
                    if 'transcript' not in content.lower() and 'call' not in content.lower():
                        future_development.append(content)
        
        # Process Search 2 results
        if 'results' in response2:
            for result in response2['results']:
                if 'content' in result and result['content']:
                    content = result['content']
                    if 'transcript' not in content.lower() and 'call' not in content.lower():
                        business_challenges.append(content)
        
        # Combine content
        combined_future = " | ".join(future_development)
        combined_challenges = " | ".join(business_challenges)
        
        # Get answers
        future_answer = response1.get('answer', '')
        challenges_answer = response2.get('answer', '')
        
        # Create structured result
        result = {
            'content': f"FUTURE BUSINESS DEVELOPMENT:\n\n{future_answer}\n\nCURRENT BUSINESS CHALLENGES & SOLUTIONS:\n\n{challenges_answer}",
            'future_development': combined_future,
            'business_challenges': combined_challenges,
            'data_source': 'Tavily_Search',
            'search_queries': [query1, query2]
        }
        
        print(f"✅ Future business development completed for {ticker}")
        print(f"🚀 Future development length: {len(combined_future)} characters")
        print(f"🔧 Business challenges length: {len(combined_challenges)} characters")
        
        return result
        
    except Exception as e:
        print(f"❌ Future business development search failed: {e}")
        return {'error': f'Future business development search failed: {str(e)}'}

def get_earnings_info_tavily_fallback(ticker, tavily_api_key):
    """
    Use Tavily search to get earnings data when FMP API fails
    Follow exact user prompts - NO current year/quarter
    
    Args:
        ticker (str): Stock ticker symbol
        tavily_api_key (str): Tavily API key
    
    Returns:
        dict: Earnings data from Tavily search
    """
    try:
        print(f"🔍 Using Tavily search for {ticker} earnings data...")
        
        # Initialize Tavily client
        client = TavilyClient(tavily_api_key)
        
        # Search 1: EXACTLY as user specified
        query1 = f"stock {ticker} current earning season summary, what is/was the business plan or development they were mentions previously"
        print(f"📊 Search 1: {query1}")
        
        response1 = client.search(
            query=query1,
            include_answer="advanced",
            topic="news",
            search_depth="advanced",
            max_results=5
        )
        
        # Search 2: EXACTLY as user specified
        query2 = f"{ticker} future business plan outlook guidance strategy growth plans in coming quarter or coming year, I want some future ahead of business plan"
        print(f"🚀 Search 2: {query2}")
        
        response2 = client.search(
            query=query2,
            include_answer="advanced",
            topic="finance",
            search_depth="advanced",
            max_results=5
        )
        
        # Extract content from both searches
        earnings_content = []
        future_plans = []
        
        # Process Search 1 results
        if 'results' in response1:
            for result in response1['results']:
                if 'content' in result and result['content']:
                    content = result['content']
                    if 'transcript' not in content.lower() and 'call' not in content.lower():
                        earnings_content.append(content)
        
        # Process Search 2 results
        if 'results' in response2:
            for result in response2['results']:
                if 'content' in result and result['content']:
                    content = result['content']
                    if 'transcript' not in content.lower() and 'call' not in content.lower():
                        future_plans.append(content)
        
        # Combine content
        combined_earnings = " | ".join(earnings_content)
        combined_future = " | ".join(future_plans)
        
        # Get answers
        earnings_answer = response1.get('answer', '')
        future_answer = response2.get('answer', '')
        
        # Create structured result
        result = {
            'symbol': ticker.upper(),
            'date': datetime.now().strftime('%Y-%m-%d'),
            'period': 'Current',
            'year': datetime.now().year,
            'content': f"EARNINGS SUMMARY (Tavily Search):\n\n{earnings_answer}\n\nFUTURE BUSINESS PLANS:\n\n{future_answer}",
            'earnings_content': combined_earnings,
            'future_plans': combined_future,
            'data_source': 'Tavily_Search',
            'search_queries': [query1, query2]
        }
        
        print(f"✅ Tavily search completed for {ticker}")
        print(f"📊 Earnings content length: {len(combined_earnings)} characters")
        print(f"🚀 Future plans length: {len(combined_future)} characters")
        
        return result
        
    except Exception as e:
        print(f"❌ Tavily search failed: {e}")
        return {'error': f'Tavily search failed: {str(e)}'}

def get_upcoming_earnings_tavily_fallback(ticker, tavily_api_key):
    """
    Use Tavily search to get earnings date when FMP API fails
    Follow exact user prompts - NO current year/quarter
    
    Args:
        ticker (str): Stock ticker symbol
        tavily_api_key (str): Tavily API key
    
    Returns:
        dict: Earnings date from Tavily search
    """
    try:
        print(f"🔍 Using Tavily search for {ticker} earnings date...")
        
        # Initialize Tavily client
        client = TavilyClient(tavily_api_key)
        
        # Search: EXACTLY as user specified
        query = f"{ticker} future business plan outlook guidance current quarter strategy growth plans"
        print(f"📅 Searching: {query}")
        
        response = client.search(
            query=query,
            include_answer="advanced",
            topic="finance",
            search_depth="advanced",
            max_results=5
        )
        
        # Extract content
        business_content = []
        if 'results' in response:
            for result in response['results']:
                if 'content' in result and result['content']:
                    content = result['content']
                    if 'transcript' not in content.lower() and 'call' not in content.lower():
                        business_content.append(content)
        
        combined_content = " | ".join(business_content)
        answer = response.get('answer', '')
        
        # Create structured result
        result = {
            'date': 'TBD',
            'eps_estimated': 'N/A',
            'revenue_estimated': 'N/A',
            'eps_actual': 'N/A',
            'revenue_actual': 'N/A',
            'last_updated': datetime.now().isoformat(),
            'data_source': 'Tavily_Search',
            'search_query': query,
            'content': answer,
            'raw_content': combined_content
        }
        
        print(f"✅ Tavily search completed for earnings date")
        print(f"📝 Content length: {len(combined_content)} characters")
        
        return result
        
    except Exception as e:
        print(f"❌ Tavily search failed: {e}")
        return {'error': f'Tavily search failed: {str(e)}'}

def get_next_earnings_date(ticker, api_key="9dfbbfa29d93f4793f246e8fb5ca5e74"):
    """
    Get the next upcoming earnings date for a ticker
    
    Args:
        ticker (str): Stock ticker symbol
        api_key (str): Financial Modeling Prep API key
    
    Returns:
        dict: Next earnings date info or error
    """
    try:
        # Get earnings calendar for next 90 days
        today = datetime.now()
        future_date = today + timedelta(days=365)
        
        url = f"https://financialmodelingprep.com/stable/earnings-calendar"
        params = {
            'from': today.strftime('%Y-%m-%d'),
            'to': future_date.strftime('%Y-%m-%d'),
            'apikey': api_key
        }
        
        print(f"📅 Getting earnings calendar from {today.strftime('%Y-%m-%d')} to {future_date.strftime('%Y-%m-%d')}...")
        
        response = requests.get(url, params=params, timeout=30)
        
        if response.status_code == 200:
            data = response.json()
            
            if data:
                # Filter for the specific ticker
                ticker_earnings = [item for item in data if item['symbol'] == ticker.upper()]
                
                if ticker_earnings:
                    # Sort by date and get the earliest (next) earnings
                    ticker_earnings.sort(key=lambda x: x['date'])
                    next_earnings = ticker_earnings[0]
                    
                    print(f"✅ Found next earnings date for {ticker}")
                    print(f"📅 Date: {next_earnings['date']}")
                    print(f"💰 EPS Estimated: {next_earnings.get('epsEstimated', 'N/A')}")
                    print(f"💰 Revenue Estimated: {next_earnings.get('revenueEstimated', 'N/A')}")
                    
                    return {
                        'date': next_earnings['date'],
                        'eps_estimated': next_earnings.get('epsEstimated'),
                        'revenue_estimated': next_earnings.get('revenueEstimated'),
                        'eps_actual': next_earnings.get('epsActual'),
                        'revenue_actual': next_earnings.get('revenueActual'),
                        'last_updated': next_earnings.get('lastUpdated')
                    }
                else:
                    print(f"❌ No upcoming earnings found for {ticker} in next 90 days")
                    return {'error': 'No upcoming earnings found in next 90 days'}
            else:
                print(f"❌ No earnings calendar data found")
                return {'error': 'No earnings calendar data available'}
                
        else:
            print(f"❌ Earnings calendar API failed: {response.status_code}")
            return {'error': f'Earnings calendar API failed: {response.status_code}'}
            
    except Exception as e:
        print(f"❌ Error getting next earnings date: {e}")
        return {'error': f'Error: {str(e)}'}

def get_latest_earnings_transcript(ticker, api_key="9dfbbfa29d93f4793f246e8fb5ca5e74"):
    """
    Get the most recent earnings transcript for a ticker
    
    Args:
        ticker (str): Stock ticker symbol (e.g., 'AAPL', 'MSFT')
        api_key (str): Financial Modeling Prep API key
    
    Returns:
        dict: Latest earnings transcript data or error message
    """
    try:
        # Step 1: Get available transcript dates for the ticker
        print(f"🔍 Getting transcript dates for {ticker}...")
        dates_url = f"https://financialmodelingprep.com/stable/earning-call-transcript-dates"
        dates_params = {
            'symbol': ticker.upper(),
            'apikey': api_key
        }
        
        dates_response = requests.get(dates_url, params=dates_params, timeout=30)
        
        if dates_response.status_code != 200:
            print(f"❌ Failed to get transcript dates: {dates_response.status_code}")
            return {'error': f'Failed to get transcript dates: {dates_response.status_code}'}
        
        dates_data = dates_response.json()
        
        if not dates_data or len(dates_data) == 0:
            print(f"❌ No transcript dates found for {ticker}")
            return {'error': 'No transcript dates available'}
        
        # Get the most recent transcript date
        latest_date = dates_data[0]  # Assuming they're sorted by date
        year = latest_date['fiscalYear']
        quarter = latest_date['quarter']
        date = latest_date['date']
        
        print(f"✅ Found latest transcript: Q{quarter} {year} ({date})")
        
        # Step 2: Get the actual transcript content
        print(f"🔍 Getting transcript content...")
        transcript_url = f"https://financialmodelingprep.com/stable/earning-call-transcript"
        transcript_params = {
            'symbol': ticker.upper(),
            'year': str(year),
            'quarter': str(quarter),
            'limit': 1,
            'apikey': api_key
        }
        
        transcript_response = requests.get(transcript_url, params=transcript_params, timeout=30)
        
        if transcript_response.status_code == 200:
            transcript_data = transcript_response.json()
            
            if transcript_data and len(transcript_data) > 0:
                transcript = transcript_data[0]
                
                print(f"✅ Found earnings transcript for {ticker}")
                print(f"📅 Date: {transcript.get('date', 'Unknown')}")
                print(f"📊 Period: {transcript.get('period', 'Unknown')}")
                print(f"📈 Year: {transcript.get('year', 'Unknown')}")
                print(f"📝 Content length: {len(transcript.get('content', ''))} characters")
                
                return transcript
            else:
                print(f"❌ No transcript content found for {ticker}")
                return {'error': 'No transcript content available'}
                
        else:
            print(f"❌ Failed to get transcript content: {transcript_response.status_code}")
            return {'error': f'Failed to get transcript content: {transcript_response.status_code}'}
            
    except Exception as e:
        print(f"❌ Error getting transcript: {e}")
        return {'error': f'Error: {str(e)}'}



In [56]:
# Usage:
earning_result = get_earnings_info_simple("CRCL")
print(f"Transcript: {earning_result['transcript'][:200]}...")
print(f"Earning Date: {earning_result['earningdate']}")
print(f"Future Development: {earning_result['future_development'][:200]}...")

🔍 Getting complete earnings info for CRCL...
📄 Step 1: Getting latest earnings transcript...
🔍 Getting transcript dates for CRCL...
❌ No transcript dates found for CRCL
⚠️ Earnings transcript not available, using Tavily search fallback...
🔍 Using Tavily search for CRCL earnings data...
📊 Search 1: stock CRCL current earning season summary, what is/was the business plan or development they were mentions previously
🚀 Search 2: CRCL future business plan outlook guidance strategy growth plans in coming quarter or coming year, I want some future ahead of business plan
✅ Tavily search completed for CRCL
📊 Earnings content length: 5242 characters
🚀 Future plans length: 1231 characters
📅 Step 3: Getting next earnings date...
📅 Getting earnings calendar from 2025-09-06 to 2026-09-06...
❌ No upcoming earnings found for CRCL in next 90 days
⚠️ Next earnings date not available, using Tavily search fallback...
🔍 Using Tavily search for CRCL earnings date...
📅 Searching: CRCL future business plan ou

In [57]:
earning_result['transcript']

"EARNINGS SUMMARY (Tavily Search):\n\nBased on the available data, Circle Internet Group (CRCL) reported Q2 2025 earnings results that showed a net loss despite overall business growth. The company's current earnings season summary indicates mixed performance with USDC stablecoin growth and an expanding ecosystem, though specific financial figures beyond the net loss are not detailed in the sources.\n\nRegarding their business plan and development strategy, CRCL has positioned itself as a key player in AI-driven energy infrastructure with several strategic initiatives. The company has established strategic partnerships with major financial services companies Fiserv and Finastra to enhance cross-border payment capabilities across financial ecosystems. Additionally, CRCL has ventured into carbon credit tokenization, which supports climate accountability in AI-driven energy projects and represents a significant expansion of their business model beyond traditional stablecoin operations.\n\

In [58]:
earning_result['earningdate']

'TBD'

In [59]:
earning_result['future_development']

'FUTURE BUSINESS DEVELOPMENT:\n\nBased on Circle Internet Group\'s recent performance and strategic positioning, the company appears well-positioned for significant expansion in the coming quarter and year. Circle reported impressive Q2 2025 results with 90% year-over-year growth in USDC circulation, 5.4x growth in on-chain transaction volume, and 53% revenue growth to $658 million, demonstrating strong momentum heading into the next periods.\n\nThe company\'s expansion strategy focuses heavily on capturing enterprise clients and diversifying beyond stablecoin operations. Circle is targeting 15,000 to 20,000 mid-sized and large clients across fintech, tokenized asset platforms, and embedded finance applications, which could generate $3-3.5 billion in recurring infrastructure revenue. This enterprise-focused approach leverages the increasing demand for regulated, on-chain infrastructure among financial institutions and fintech companies.\n\nCircle\'s growth projections suggest the compa